# M5 · Best Ensemble Run v2 (Per-Segment Full FT Params)

Runs the best per-segment Chronos-2 ensemble found by `hpo_ft_ensembles_v6`.

Each segment has its own `(CL, ft_steps, ft_mode, ft_lr, ft_bs, cov_type)`.
Paste the best-trial values from the v6 HPO study into the `config` cell.

The `_make_seg_tag` cache key is identical to v6 — any segment forecast already
computed during HPO is reused automatically (instant parquet read).

In [ ]:
import sys
sys.path.append("/home/nmwamsojo/tsfm-explo/src/jobs/")

import os
import gc
import ctypes

import torch
import pandas as pd
from IPython.display import display

from m5_dataprep import M5DataPipeline
from m5_exploration import M5ExplorationSuite, DEFAULT_CHRONOS_CONFIG
from m5_evaluator import M5Evaluator

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({props.total_memory // 1024**3} GB)")

In [ ]:
os.environ["OMP_NUM_THREADS"]        = "4"
os.environ["MKL_NUM_THREADS"]        = "4"
os.environ["OPENBLAS_NUM_THREADS"]   = "4"
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"]    = "4"
os.environ["CUDA_VISIBLE_DEVICES"]   = "0,1"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Inference device: {DEVICE}")

# Config — paste HPO v6 best trial here

In [ ]:
DATA_PATH     = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH  = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"
DATA_TAG      = "sales_only"

CUTOFF_DAY = (
    pd.to_datetime("2016-05-22") - pd.Timedelta(days=28)
).strftime("%Y-%m-%d")
print(f"Cutoff day : {CUTOFF_DAY}")
print(f"Data tag   : {DATA_TAG}")

# ── Segmentation ────────────────────────────────────────────────────────────────────────────
# "weight_segs"     → segments: "Low", "Medium-Low", "Medium-High", "High"
# "smoothness_segs" → segments: "Smooth", "Erratic", "Intermittent", "Lumpy"
SEG_COLUMNS = "smoothness_segs"

# ── Per-segment best params — paste from HPO v6 results ─────────────────────────────
# Fill in the best values from hpo_ft_ensembles_v6 study.best_trial
BEST_PARAMS = {
    "Smooth":       {"cl": 256, "ft_steps": 0,   "ft_mode": "lora", "ft_lr": 1e-4,  "ft_bs": 128, "cov_type": "is_weekend"},
    "Erratic":      {"cl": 128, "ft_steps": 0,   "ft_mode": "lora", "ft_lr": 1e-4,  "ft_bs": 128, "cov_type": "is_weekend"},
    "Intermittent": {"cl": 64,  "ft_steps": 0,   "ft_mode": "lora", "ft_lr": 1e-4,  "ft_bs": 128, "cov_type": "is_weekend"},
    "Lumpy":        {"cl": 32,  "ft_steps": 0,   "ft_mode": "lora", "ft_lr": 1e-4,  "ft_bs": 128, "cov_type": "is_weekend"},
}
# If using weight_segs, replace the dict keys above with:
# "Low", "Medium-Low", "Medium-High", "High"

BATCH_SIZE = 256
FORCE_RUN  = False   # True → re-run even if a cached forecast exists

# ── Covariate options — must match hpo_ft_ensembles_v6 exactly ───────────────────────
COVARIATE_OPTIONS = {
    "is_weekend":  ["is_friday", "is_saturday", "is_sunday"],
    "event":       ["event_name_1", "event_type_1"],
    "price":       ["sell_price"],
    "snap":        ["snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_event":  ["is_friday", "is_saturday", "is_sunday", "event_name_1", "event_type_1"],
    "is_weekend_price":  ["is_friday", "is_saturday", "is_sunday", "sell_price"],
    "event_price": ["event_name_1", "event_type_1", "sell_price"],
    "is_weekend_snap":       ["is_friday", "is_saturday", "is_sunday", "snap_CA", "snap_TX", "snap_WI"],
    "event_snap":            ["event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"],
    "price_snap":            ["sell_price", "snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_event_snap": ["is_friday", "is_saturday", "is_sunday", "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_price_snap": ["is_friday", "is_saturday", "is_sunday", "sell_price", "snap_CA", "snap_TX", "snap_WI"],
    "event_price_snap":      ["event_name_1", "event_type_1", "sell_price", "snap_CA", "snap_TX", "snap_WI"],
    "all": ["is_friday", "is_saturday", "is_sunday", "event_name_1", "event_type_1", "sell_price", "snap_CA", "snap_TX", "snap_WI"],
}

WRAPPER = {
    "eval_metric":          "RMSSE",
    "enable_ensemble":      False,
    "skip_model_selection": True,
    "verbosity":            1,
}

# Validate BEST_PARAMS keys match the chosen segmentation
SEG_NAMES_MAP = {
    "weight_segs":     ["Low", "Medium-Low", "Medium-High", "High"],
    "smoothness_segs": ["Smooth", "Erratic", "Intermittent", "Lumpy"],
}
expected_segs = SEG_NAMES_MAP[SEG_COLUMNS]
missing = [s for s in expected_segs if s not in BEST_PARAMS]
unknown = [s for s in BEST_PARAMS if s not in expected_segs]
if missing:
    raise ValueError(f"BEST_PARAMS missing segments: {missing}")
if unknown:
    raise ValueError(f"BEST_PARAMS has unexpected segments for {SEG_COLUMNS}: {unknown}")

print(f"\nSegmentation  : {SEG_COLUMNS}")
seg_scheme = SEG_COLUMNS.replace("_segs", "")
print(f"Scheme        : {seg_scheme}")
print(f"\n{'Segment':>14}  {'CL':>5}  {'ft_steps':>8}  {'ft_mode':>8}  {'ft_lr':>8}  {'ft_bs':>6}  cov_type")
print("-" * 75)
for seg in expected_segs:
    p = BEST_PARAMS[seg]
    print(f"  {seg:>12}  {p['cl']:>5}  {p['ft_steps']:>8}  {p['ft_mode']:>8}  {p['ft_lr']:>8.0e}  {p['ft_bs']:>6}  {p['cov_type']}")

# Data

In [ ]:
pipeline = M5DataPipeline(config={"tag": DATA_TAG})
hist_df, hist_df_trimmed, future_df, static_df, weights_scales = (
    pipeline.get_prepared_data(DATA_PATH, CUTOFF_DAY, level=12, force_reprepare=False)
)

print(f"hist_df_trimmed : {hist_df_trimmed.shape}")
print(f"future_df       : {future_df.shape}")

del pipeline
gc.collect()

In [ ]:
def _wide_to_long(gt_wide: pd.DataFrame, calendar_path: str) -> pd.DataFrame:
    if "id" not in gt_wide.columns:
        gt_wide["id"] = gt_wide["item_id"] + "_" + gt_wide["store_id"] + "_evaluation"
    day_cols = [c for c in gt_wide.columns if c.startswith("d_")]
    long = gt_wide.melt(id_vars=["id"], value_vars=day_cols,
                        var_name="d", value_name="sales_quantity")
    cal = pd.read_csv(calendar_path, usecols=["d", "date"])
    cal["date"] = pd.to_datetime(cal["date"])
    long = long.merge(cal, on="d", how="left").drop(columns=["d"])
    long["id"] = long["id"].str.replace("_evaluation", "", regex=False)
    return long[["id", "date", "sales_quantity"]]


if CUTOFF_DAY == "2016-05-22":
    _eval_raw = pd.read_csv(ACTUALS_PATH)
    df_actual = _wide_to_long(_eval_raw, CALENDAR_PATH)
    del _eval_raw
else:
    df_actual = (
        pd.read_parquet(DATA_PATH, columns=["id", "date", "sold"])
        .rename(columns={"sold": "sales_quantity"})
    )
    df_actual["id"] = (
        df_actual["id"].astype(str)
        .str.replace("_evaluation", "", regex=False)
        .str.replace("_validation",  "", regex=False)
    )
print(f"Actuals : {df_actual.shape}  |  {df_actual['date'].min().date()} → {df_actual['date'].max().date()}")

# Evaluator & Suite

In [ ]:
evaluator = M5Evaluator(
    raw_train_df     = hist_df,
    trimmed_train_df = hist_df_trimmed,
    static_df        = static_df,
    weights_df       = weights_scales,
    target_col       = "sales_quantity",
    price_col        = "sell_price",
)

# Module-level suite — reused across all segments (NOT re-created per segment).
# _run_one_segment in the run cell accesses this directly via global lookup.
suite = M5ExplorationSuite(
    horizon  = 28,
    ag_path  = "/mnt/lab/nmwamsojo/autogluon_models/explorations",
    base_dir = "/mnt/lab/nmwamsojo/prepared_data",
)
print("Evaluator and suite ready.")

# Segment ID Map

In [ ]:
quantile_labels = ["Low", "Medium-Low", "Medium-High", "High"]
weights_scales["weight_segs"] = pd.qcut(
    weights_scales["weight"],
    q=4,
    labels=quantile_labels,
)

In [ ]:
UNDEFINED_IN = {
    "weight_segs":     "Low",
    "smoothness_segs": "Lumpy",
}
undef_target = UNDEFINED_IN[SEG_COLUMNS]

if SEG_COLUMNS in hist_df_trimmed.columns:
    _src = hist_df_trimmed[["id", SEG_COLUMNS]].drop_duplicates()
else:
    _src = (
        weights_scales[weights_scales["level"] == 12][["id", SEG_COLUMNS]]
        .dropna(subset=[SEG_COLUMNS])
    )

seg_id_map: dict[str, list] = {
    seg: _src[_src[SEG_COLUMNS] == seg]["id"].tolist()
    for seg in expected_segs
}
seg_id_map[undef_target].extend(_src[_src[SEG_COLUMNS] == "Undefined"]["id"].tolist())
del _src

total = sum(len(v) for v in seg_id_map.values())
print(f"\n[{seg_scheme}]")
for seg in expected_segs:
    print(f"  {seg:>14}: {len(seg_id_map[seg]):>6,}  ({100*len(seg_id_map[seg])/total:.1f}%)")
print(f"  {'TOTAL':>14}: {total:>6,}")

# Run Per-Segment Ensemble

In [ ]:
def _make_seg_tag(seg_scheme: str, cl: int,
                  ft_steps: int, ft_mode: str, ft_lr: float,
                  ft_bs: int, cov_type: str) -> str:
    """Cache key — identical to hpo_ft_ensembles_v6 for cross-notebook cache reuse."""
    if ft_steps == 0:
        return f"hpo_zs_{seg_scheme}_cl{cl}_{cov_type}"
    lr_str = f"{ft_lr:.0e}".replace("-0", "-")
    return (
        f"hpo_ft_{seg_scheme}_cl{cl}"
        f"_steps{ft_steps}_{ft_mode}_lr{lr_str}_bs{ft_bs}_{cov_type}"
    )


CFG_CHRONOS_BASE = {
    **DEFAULT_CHRONOS_CONFIG,
    "use_static": False,
}

print("Helper functions ready.")

In [ ]:
seg_forecasts = []

for seg_name in expected_segs:
    p       = BEST_PARAMS[seg_name]
    seg_tag = _make_seg_tag(
        seg_scheme, p["cl"], p["ft_steps"], p["ft_mode"], p["ft_lr"], p["ft_bs"], p["cov_type"]
    )
    print(f"\n[{seg_name}]  CL={p['cl']}  ft_steps={p['ft_steps']}  "
          f"mode={p['ft_mode']}  cov={p['cov_type']}  tag={seg_tag}")

    # Clear TSDF cache before each segment — releases previous segment's
    # TimeSeriesDataFrame (~1-3 GB) before building the new one.
    suite._cached_tsdf.clear()
    suite._cached_future.clear()
    gc.collect(2)
    torch.cuda.empty_cache()

    cfg = {
        **CFG_CHRONOS_BASE,
        "context_length":       p["cl"],
        "fine_tune_steps":      p["ft_steps"],
        "fine_tune_mode":       p["ft_mode"],
        "fine_tune_lr":         p["ft_lr"],
        "fine_tune_batch_size": p["ft_bs"],
        "batch_size":           BATCH_SIZE,
        "known_cov_cols":       COVARIATE_OPTIONS[p["cov_type"]],
        "device":               DEVICE,
    }

    try:
        fcst = suite.run(
            hist_df      = hist_df_trimmed,
            future_df    = future_df,
            static_df    = static_df,
            model        = "Chronos2",
            exp_config   = cfg,
            exp_tag      = seg_tag,
            data_tag     = DATA_TAG,
            cutoff_day   = CUTOFF_DAY,
            wrapper_dict = WRAPPER,
            force_run    = FORCE_RUN,
        )
        seg_ids  = set(seg_id_map[seg_name])
        filtered = fcst[fcst["id"].isin(seg_ids)].copy()
        del fcst
        seg_forecasts.append(filtered)
        del filtered
        print(f"  ✓ {len(seg_forecasts[-1]):,} rows added")

    except Exception as exc:
        print(f"  [FAILED] {exc}")

    finally:
        gc.collect(2)
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass

if len(seg_forecasts) == len(expected_segs):
    ensemble_fcst = pd.concat(seg_forecasts, ignore_index=True)
    del seg_forecasts
    print(f"\nEnsemble : {ensemble_fcst.shape}")

    metrics = evaluator.evaluate_all(ensemble_fcst, df_actual)
    del ensemble_fcst
    print(f"\nWRMSSE = {metrics['WRMSSE']:.4f}")
    print(f"WAPE   = {metrics.get('WAPE_L12', float('nan')):.2%}")
else:
    print(f"\n[WARNING] Only {len(seg_forecasts)}/{len(expected_segs)} segments completed — skipping evaluation.")
    metrics = {}

# Results

In [ ]:
if metrics:
    print("\nFull metrics:")
    for k, v in sorted(metrics.items()):
        print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v}")